# Notebook 01 — BGP Troubleshooting ReAct Agent

> **Use case:** A network operator reports *site A cannot reach site B*. We build a tiny **ReAct** (Reason + Act) agent that uses **read-only tools** on a mocked network to diagnose the root cause.

**What you will learn**
- How an agent **loop** works in practice (thought → action → observation → ...).
- How to expose **typed tools** to an agent.
- How to design tools that are **safe** (read-only, scoped).
- How to swap a mock 'LLM' for a real one (Ollama / OpenAI) at the end.

The notebook runs **fully offline** — no API key required.

## 1. Mock network state

We pretend to operate a small 4-router fabric. One fault has been planted: on `r2` the inbound prefix-list `FROM_R1` was tightened 12 minutes ago, which removed `10.1.0.0/24` (site A) from `r2`'s table, and the BGP session toward `r1` is now `Idle`.

In [1]:
from datetime import datetime, timedelta

NOW = datetime(2026, 5, 29, 14, 30)

DEVICES = {
    "r1": {
        "interfaces": {
            "eth0": {"status": "up", "ip": "10.0.12.1/30"},
            "lo0":  {"status": "up", "ip": "10.1.0.1/32"},
        },
        "bgp": [
            {"peer": "10.0.12.2", "asn": 65002, "state": "Idle", "uptime": "0s"},
        ],
        "routes": {
            "10.1.0.0/24": {"next_hop": "connected", "protocol": "local"},
        },
    },
    "r2": {
        "interfaces": {
            "eth0": {"status": "up", "ip": "10.0.12.2/30"},
            "eth1": {"status": "up", "ip": "10.0.23.2/30"},
            "lo0":  {"status": "up", "ip": "10.2.0.1/32"},
        },
        "bgp": [
            {"peer": "10.0.12.1", "asn": 65001, "state": "Idle",        "uptime": "0s"},
            {"peer": "10.0.23.3", "asn": 65003, "state": "Established", "uptime": "3d"},
        ],
        "routes": {
            "10.2.0.0/24": {"next_hop": "connected", "protocol": "local"},
            "10.3.0.0/24": {"next_hop": "10.0.23.3", "protocol": "bgp"},
            # NOTE: 10.1.0.0/24 (site A) is MISSING here -> this is the symptom
        },
    },
    "r3": {
        "interfaces": {"eth0": {"status": "up", "ip": "10.0.23.3/30"}},
        "bgp":        [{"peer": "10.0.23.2", "asn": 65002, "state": "Established", "uptime": "3d"}],
        "routes":     {"10.3.0.0/24": {"next_hop": "connected", "protocol": "local"}},
    },
}

CHANGES = [
    {"when": NOW - timedelta(minutes=12), "device": "r2", "who": "alice",
     "summary": "Tightened inbound prefix-list FROM_R1 (removed 10.1.0.0/24)"},
    {"when": NOW - timedelta(days=2),    "device": "r3", "who": "bob",
     "summary": "Added loopback lo0"},
]

print("Topology loaded:", list(DEVICES))
print("Recent changes:", len(CHANGES))

Topology loaded: ['r1', 'r2', 'r3']
Recent changes: 2


## 2. Tools the agent can call

We expose **four read-only tools**. Each one has a clear name, a typed input schema, and returns a small structured observation. This is exactly what a real MCP server would expose — we keep it in-process for clarity.

In [2]:
import json
from typing import Any, Callable

def get_bgp_neighbors(device: str) -> dict:
    """Return BGP neighbors and their state for a device."""
    if device not in DEVICES:
        return {"error": f"unknown device {device}"}
    return {"device": device, "neighbors": DEVICES[device]["bgp"]}

def get_interface(device: str, name: str) -> dict:
    """Return interface status (up/down) and IP for a device."""
    intfs = DEVICES.get(device, {}).get("interfaces", {})
    if name not in intfs:
        return {"error": f"no interface {name} on {device}"}
    return {"device": device, "interface": name, **intfs[name]}

def get_route_to(device: str, prefix: str) -> dict:
    """Return the route a device has for a given destination prefix (or None)."""
    routes = DEVICES.get(device, {}).get("routes", {})
    return {"device": device, "prefix": prefix, "route": routes.get(prefix)}

def get_recent_changes(device: str, minutes: int = 60) -> dict:
    """Return change-management records for a device in the last N minutes."""
    cutoff = NOW - timedelta(minutes=minutes)
    hits = [c for c in CHANGES if c["device"] == device and c["when"] >= cutoff]
    return {"device": device, "window_minutes": minutes,
            "changes": [{**c, "when": c["when"].isoformat()} for c in hits]}

TOOLS: dict[str, dict[str, Any]] = {
    "get_bgp_neighbors": {"fn": get_bgp_neighbors,
                         "schema": {"device": "str"},
                         "doc": "BGP neighbors + state on a device."},
    "get_interface":     {"fn": get_interface,
                         "schema": {"device": "str", "name": "str"},
                         "doc": "Interface up/down + IP."},
    "get_route_to":      {"fn": get_route_to,
                         "schema": {"device": "str", "prefix": "str"},
                         "doc": "Best route on device for prefix (None if absent)."},
    "get_recent_changes":{"fn": get_recent_changes,
                         "schema": {"device": "str", "minutes": "int"},
                         "doc": "Recent change-management entries for device."},
}

def tool_catalogue() -> str:
    return "\n".join(
        f"- {n}({', '.join(f'{k}: {v}' for k,v in t['schema'].items())}) -> {t['doc']}"
        for n, t in TOOLS.items()
    )

print(tool_catalogue())

- get_bgp_neighbors(device: str) -> BGP neighbors + state on a device.
- get_interface(device: str, name: str) -> Interface up/down + IP.
- get_route_to(device: str, prefix: str) -> Best route on device for prefix (None if absent).
- get_recent_changes(device: str, minutes: int) -> Recent change-management entries for device.


## 3. A *mock* LLM (so the notebook runs anywhere)

A real ReAct agent calls an LLM at every step. To keep this notebook self-contained, we replace the LLM with a tiny **rule-based reasoner** that follows the same protocol an LLM would: read the trajectory, return either an action or a final answer.

At the end of the notebook we show how to plug a **real** LLM in its place.

In [3]:
def mock_llm(goal: str, trajectory: list[dict]) -> dict:
    """Toy planner that mimics what an LLM would decide step by step."""
    seen = {s.get("action", {}).get("tool") for s in trajectory if "action" in s}

    if "get_route_to" not in seen:
        return {"thought": "Site A is 10.1.0.0/24. Check if r2 has a route to it.",
                "action": {"tool": "get_route_to",
                           "args": {"device": "r2", "prefix": "10.1.0.0/24"}}}

    if "get_bgp_neighbors" not in seen:
        return {"thought": "No route on r2. Maybe the BGP session toward r1 is down.",
                "action": {"tool": "get_bgp_neighbors", "args": {"device": "r2"}}}

    if "get_interface" not in seen:
        return {"thought": "Peer 10.0.12.1 is Idle. Is the underlying link eth0 up?",
                "action": {"tool": "get_interface",
                           "args": {"device": "r2", "name": "eth0"}}}

    if "get_recent_changes" not in seen:
        return {"thought": "Link is up but BGP is Idle. Check recent changes on r2.",
                "action": {"tool": "get_recent_changes",
                           "args": {"device": "r2", "minutes": 60}}}

    return {
        "thought": "Recent change tightened a prefix-list removing 10.1.0.0/24. Likely cause.",
        "final": (
            "ROOT CAUSE: 12 minutes ago alice modified inbound prefix-list FROM_R1 on r2 "
            "and removed 10.1.0.0/24, which is why r2 has no route to site A and the BGP "
            "session toward r1 stayed Idle from a policy standpoint.\n"
            "RECOMMENDED FIX (requires human approval): revert prefix-list FROM_R1 to its "
            "previous revision, then clear the BGP session toward 10.0.12.1.\n"
            "CONFIDENCE: high — single recent change, matches symptom precisely."
        ),
    }

## 4. The ReAct loop

This is the heart of every agent:
1. ask the model for the next *thought + action* (or a final answer),
2. execute the action against our tool registry,
3. append the observation to the trajectory,
4. stop on a final answer, on too many steps, or on too many errors.

This is exactly what frameworks like LangGraph or NeMo Agent Toolkit do — just hidden behind nicer APIs.

In [4]:
MAX_STEPS = 8

def run_react(goal: str, llm: Callable[[str, list[dict]], dict]) -> dict:
    trajectory: list[dict] = []
    for step in range(1, MAX_STEPS + 1):
        decision = llm(goal, trajectory)

        if "final" in decision:
            trajectory.append({"step": step, "thought": decision["thought"],
                               "final": decision["final"]})
            return {"status": "done", "steps": step,
                    "trajectory": trajectory, "answer": decision["final"]}

        action = decision["action"]
        tool = TOOLS.get(action["tool"])
        if tool is None:
            obs = {"error": f"unknown tool {action['tool']}"}
        else:
            try:
                obs = tool["fn"](**action["args"])
            except Exception as e:
                obs = {"error": f"{type(e).__name__}: {e}"}

        trajectory.append({"step": step, "thought": decision["thought"],
                           "action": action, "observation": obs})

    return {"status": "budget_exhausted", "steps": MAX_STEPS,
            "trajectory": trajectory, "answer": None}


def print_trace(result: dict) -> None:
    print(f"=== Agent finished: {result['status']} in {result['steps']} step(s) ===\n")
    for s in result["trajectory"]:
        print(f"-- step {s['step']} --")
        print(f"THOUGHT: {s['thought']}")
        if "action" in s:
            print(f"ACTION : {s['action']['tool']}({s['action']['args']})")
            print(f"OBSERV : {json.dumps(s['observation'], default=str)}")
        if "final" in s:
            print("FINAL  :")
            print(s["final"])
        print()

In [5]:
GOAL = "Clients behind r2 cannot reach site A (10.1.0.0/24). Find the root cause and propose a fix."
result = run_react(GOAL, mock_llm)
print_trace(result)

=== Agent finished: done in 5 step(s) ===

-- step 1 --
THOUGHT: Site A is 10.1.0.0/24. Check if r2 has a route to it.
ACTION : get_route_to({'device': 'r2', 'prefix': '10.1.0.0/24'})
OBSERV : {"device": "r2", "prefix": "10.1.0.0/24", "route": null}

-- step 2 --
THOUGHT: No route on r2. Maybe the BGP session toward r1 is down.
ACTION : get_bgp_neighbors({'device': 'r2'})
OBSERV : {"device": "r2", "neighbors": [{"peer": "10.0.12.1", "asn": 65001, "state": "Idle", "uptime": "0s"}, {"peer": "10.0.23.3", "asn": 65003, "state": "Established", "uptime": "3d"}]}

-- step 3 --
THOUGHT: Peer 10.0.12.1 is Idle. Is the underlying link eth0 up?
ACTION : get_interface({'device': 'r2', 'name': 'eth0'})
OBSERV : {"device": "r2", "interface": "eth0", "status": "up", "ip": "10.0.12.2/30"}

-- step 4 --
THOUGHT: Link is up but BGP is Idle. Check recent changes on r2.
ACTION : get_recent_changes({'device': 'r2', 'minutes': 60})
OBSERV : {"device": "r2", "window_minutes": 60, "changes": [{"when": "2026-0

## 5. Optional — plug a real LLM via Ollama

The cell below shows how to **drop in a real LLM** without changing anything else. It needs a local [Ollama](https://ollama.com) server with a model that supports JSON output (e.g. `llama3.1:8b`). If Ollama is not reachable the cell prints a hint and skips — the rest of the notebook is unaffected.

In production you would replace `mock_llm` with this `real_llm` and add proper retries, timeouts, and a system prompt that mentions the tool catalogue. Notice the **agent loop is unchanged** — only the brain is swapped.

In [ ]:
SYSTEM_PROMPT = f"""You are a senior network engineer agent. Diagnose the user's goal using ONLY the tools below.
Return STRICT JSON each turn — either:
  {{"thought": "...", "action": {{"tool": "<name>", "args": {{...}}}}}}
or when you are confident:
  {{"thought": "...", "final": "<root cause + recommended fix>"}}

Available tools:
{tool_catalogue()}
"""

def real_llm(goal: str, trajectory: list[dict]) -> dict:
    try:
        from openai import OpenAI
    except ImportError:
        print("openai package not installed — skipping real_llm. Run: pip install openai")
        return {"thought": "openai client missing", "final": "(skipped real LLM)"}

    client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": f"GOAL: {goal}\nTRAJECTORY: {json.dumps(trajectory, default=str)}"},
    ]
    try:
        resp = client.chat.completions.create(
            model="llama3.1:8b",
            messages=messages,
            response_format={"type": "json_object"},
            temperature=0.1,
            timeout=20,
        )
        return json.loads(resp.choices[0].message.content)
    except Exception as e:
        print(f"(Ollama unreachable — {e}; skipping real LLM)")
        return {"thought": "ollama unreachable", "final": "(skipped real LLM)"}

# Uncomment to try with a real model:
 real_result = run_react(GOAL, real_llm)
 print_trace(real_result)
print("real_llm() is ready — uncomment the two lines above to run it against Ollama.")

real_llm() is ready — uncomment the two lines above to run it against Ollama.


## Exercises

1. **Add a new fault.** Set `r3`'s `eth0` to `down` and re-run. Does the mock LLM diagnose it? Why not? Improve it.
2. **Add a write tool.** Implement `clear_bgp_session(device, peer)` that mutates `DEVICES`. Then put it behind a **human approval** gate (`input("approve? [y/N]")`) before executing.
3. **Budgets.** Lower `MAX_STEPS` to `2` — what does the trace look like? How would you surface this to the operator?
4. **Real LLM.** Install Ollama, pull `llama3.1:8b`, uncomment the last cell. Compare its trajectory to the rule-based one.
5. **Audit log.** Extend `run_react` to write each step as a JSON line to `audit.log`. That is the bare minimum auditability requirement of Chapter 12.